In [119]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [73]:
iris = load_iris()
X = iris.data
y = iris.target

In [74]:
print(f"No of samples: {X.shape[0]}")
print(f"No of features: {X.shape[1]}")
print("Class distribution:\n")
for i in np.unique(y):
  print(f"{iris.target_names[i]} : {np.sum(y==i)}")

No of samples: 150
No of features: 4
Class distribution:

setosa : 50
versicolor : 50
virginica : 50


In [108]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=40, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [109]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    metric='euclidean'
)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Accuracy: 0.9333


In [110]:
def evaluate(y_test, y_pred):
  return [
      accuracy_score(y_test, y_pred),
      precision_score(y_test, y_pred, average='macro', zero_division=0),
      recall_score(y_test, y_pred, average='macro', zero_division=0),
      f1_score(y_test, y_pred, average='macro', zero_division=0)
  ]

In [111]:
results_k = []
k_values = [1,3,5,7,9,11]
for k in k_values:
  knn=KNeighborsClassifier(
      n_neighbors=k,
      metric='euclidean'
  )
  knn.fit(X_train, y_train)
  y_pred = knn.predict(X_test)
  results_k.append([k]+evaluate(y_test, y_pred))
df_k = pd.DataFrame(results_k, columns=[
    'k', 'accuracy', 'precision', 'recall', 'f1 score'
])

In [112]:
df_k

,k,accuracy,precision,recall,f1 score
0,1,0.933333,0.944444,0.933333,0.932660
1,3,0.933333,0.944444,0.933333,0.932660
2,5,0.933333,0.944444,0.933333,0.932660
3,7,0.966667,0.969697,0.966667,0.966583
4,9,0.966667,0.969697,0.966667,0.966583
5,11,0.966667,0.969697,0.966667,0.966583


In [113]:
best_k = df_k[df_k['accuracy']==np.max(df_k['accuracy'])]
best_k.values[:, 0]

array([ 7.,  9., 11.])

In [114]:
results_d = []
dist = ['euclidean', 'manhattan', 'chebyshev']
for d in dist:
  knn = KNeighborsClassifier(
      n_neighbors=int(best_k.values[0,0]),
      metric=d
  )
  knn.fit(X_train, y_train)
  y_pred = knn.predict(X_test)
  results_d.append([d]+evaluate(y_test, y_pred))
df_d=pd.DataFrame(results_d, columns=[
    'distance', 'accuracy', 'precision', 'recall', 'f1 score'
])

In [115]:
df_d

,distance,accuracy,precision,recall,f1 score
0,euclidean,0.966667,0.969697,0.966667,0.966583
1,manhattan,0.933333,0.944444,0.933333,0.932660
2,chebyshev,0.933333,0.944444,0.933333,0.932660


In [116]:
comparison = []
for k in k_values:
  for d in dist:
    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric=d
    )
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    comparison.append({
        'k':k,
        'distance':d,
        'accuracy':accuracy
    })

In [117]:
comparison_df = pd.DataFrame(comparison)

comparison_df_pivot = pd.pivot_table(
    comparison_df,
    index='k',
    columns='distance',
    values='accuracy'
)

comparison_df_pivot

distance,chebyshev,euclidean,manhattan
k,,,
1,0.933333,0.933333,0.933333
3,0.933333,0.933333,0.933333
5,0.933333,0.933333,0.933333
7,0.933333,0.966667,0.933333
9,0.933333,0.966667,0.966667
11,0.933333,0.966667,0.933333


In [118]:
best_performing_combination = comparison_df[comparison_df['accuracy']==np.max(comparison_df['accuracy'])]
best_performing_combination

,k,distance,accuracy
9,7,euclidean,0.966667
12,9,euclidean,0.966667
13,9,manhattan,0.966667
15,11,euclidean,0.966667
